# Deepfake Detection Dataset 2026: EDA

Use this notebook to inspect the Kaggle metadata, validate label and split distributions, and generate lightweight exploratory plots.

Run the Kaggle download first if `data/raw/FINAL_DATASET.csv` is missing:

```powershell
python scripts/download_kaggle_dataset.py
```

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from deepfake_detection.config import load_config, resolve_path
from deepfake_detection.data import normalize_splits, read_metadata

config = load_config(PROJECT_ROOT / "configs/default.yaml")
dataset_cfg = config["dataset"]
reports_dir = resolve_path(config["reports"]["output_dir"])
reports_dir.mkdir(parents=True, exist_ok=True)
config

In [ ]:
df = read_metadata(dataset_cfg["csv_path"])
df = normalize_splits(
    df,
    label_column=dataset_cfg["label_column"],
    split_column=dataset_cfg["split_column"],
    seed=config["training"]["seed"],
)

print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
summary_columns = [
    "label",
    dataset_cfg["label_column"],
    dataset_cfg["split_column"],
    "gender",
    "age_group",
    "image_quality",
    "source",
    "fake_method",
    "detection_difficulty",
]

for column in summary_columns:
    if column in df.columns:
        display(df[column].value_counts(dropna=False).rename_axis(column).to_frame("count"))

In [ ]:
import matplotlib.pyplot as plt

plot_columns = ["label", dataset_cfg["split_column"], "gender", "age_group", "image_quality"]
fig, axes = plt.subplots(len(plot_columns), 1, figsize=(8, 4 * len(plot_columns)))

for ax, column in zip(axes, plot_columns):
    if column in df.columns:
        df[column].value_counts(dropna=False).plot(kind="bar", ax=ax, title=f"{column} distribution")
        ax.set_xlabel(column)
        ax.set_ylabel("count")

plt.tight_layout()

In [ ]:
if {"label", dataset_cfg["split_column"]}.issubset(df.columns):
    split_label_counts = df.groupby([dataset_cfg["split_column"], "label"]).size().unstack(fill_value=0)
    display(split_label_counts)
    split_label_counts.plot(kind="bar", stacked=True, figsize=(8, 4), title="Labels by split")

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
from deepfake_detection.eda import run_eda

summary = run_eda(PROJECT_ROOT / "configs/default.yaml")
summary